In [1]:
import os
import io
import pandas as pd
import boto3
from botocore.exceptions import ClientError
from dotenv import load_dotenv

In [4]:
s3_client = boto3.client(
    "s3",
    aws_access_key_id=os.getenv("AWS_ACCESS_KEY_ID"),
    aws_secret_access_key=os.getenv("AWS_SECRET_ACCESS_KEY"),
    aws_session_token=os.getenv("AWS_SESSION_TOKEN"),
    region_name=os.getenv("AWS_DEFAULT_REGION"),
)

bucket_name = "practice-datalake-carlos-jaramillo"

def mostrar_tamaños_s3(bucket, prefix):
    """
    Lista los archivos de una zona S3 y muestra su tamaño en bytes.
    """
    try:
        response = s3_client.list_objects_v2(
            Bucket=bucket,
            Prefix=prefix
        )

        if "Contents" not in response:
            print(f"No se encontraron archivos en s3://{bucket}/{prefix}")
            return 0

        total_bytes = 0

        print(f"\nZona: s3://{bucket}/{prefix}")

        for obj in response["Contents"]:
            # Evitar mostrar el prefijo como si fuera un archivo
            if obj["Key"].endswith("/"):
                continue

            size = obj["Size"]
            total_bytes += size

            print(
                f"Archivo: {obj['Key']} | "
                f"Tamaño: {size:,} bytes"
            )

        print(f"Total: {total_bytes:,} bytes")

        return total_bytes, size, obj['Key']

    except Exception as e:
        print(f"Error al consultar S3: {e}")
        return 0


# Tamaño de los archivos en cada zona
raw_size, raw_size_value, raw_name = mostrar_tamaños_s3(
    bucket_name,
    "raw_zone/"
)
optimized_size, optimized_size_value, optimized_name = mostrar_tamaños_s3(
    bucket_name,
    "optimized_zone/"
)
consumption_size, consumption_size_value, consumption_name = mostrar_tamaños_s3(
    bucket_name,
    "consumption_zone/"
)


Zona: s3://practice-datalake-carlos-jaramillo/raw_zone/
Archivo: raw_zone/restaurants_raw.csv | Tamaño: 19,061 bytes
Total: 19,061 bytes

Zona: s3://practice-datalake-carlos-jaramillo/optimized_zone/
Archivo: optimized_zone/restaurants_processed.parquet | Tamaño: 6,960 bytes
Total: 6,960 bytes

Zona: s3://practice-datalake-carlos-jaramillo/consumption_zone/
Archivo: consumption_zone/category_kpis.parquet | Tamaño: 3,867 bytes
Total: 3,867 bytes


In [5]:
# Print comparison
print("--- File and DataFrame Comparison ---")
print("\nCSV File:")
print(f"  - File Path: {raw_name}")
print(f"  - Size on disk: {raw_size_value / 1024:.2f} KB")
 
print("\nParquet File:")
print(f"  - File Path: {optimized_name}")
print(f"  - Size on disk: {optimized_size_value / 1024:.2f} KB")
 
# Highlight the size difference
size_difference = (raw_size_value - optimized_size_value) / raw_size_value * 100
print(f"\nNote: The Parquet file is {size_difference:.2f}% smaller than the CSV file.")

--- File and DataFrame Comparison ---

CSV File:
  - File Path: raw_zone/restaurants_raw.csv
  - Size on disk: 18.61 KB

Parquet File:
  - File Path: optimized_zone/restaurants_processed.parquet
  - Size on disk: 6.80 KB

Note: The Parquet file is 63.49% smaller than the CSV file.
